In [ ]:
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *
from src.data.plots import image_category

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

In [ ]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(CSV_DIR/"data/metadata.csv")

df["boxes"] = df["boxes"].apply(json.loads)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,6))

fire_split = df["label"].map({0: "Non-fire/smoke", 1: "Fire/Smoke"}).value_counts()
labels = [f"{label} ({count:,})" for label, count in fire_split.items()]

fire_split.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
    ax=axes[0]
)
axes[0].set_title("Images With and Without Fire/Smoke")

df["category"] = df["boxes"].apply(image_category)
cat_counts = df.loc[
    df["category"] != "neither", # Exclude neither because its already shown in fire/non-fire
    "category"
].value_counts()
labels = [f"{label.capitalize()} ({count})" for label, count in cat_counts.items()]

cat_counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
    ax=axes[1]
)
axes[1].set_title("Distribution of Fire and Smoke Images")

plt.tight_layout()
fig.savefig(PLOT_DIR / "image_category_distribution.svg")
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))

split_counts = df["split"].value_counts()
labels = [f"{label.capitalize()} ({count:,})" for label, count in split_counts.items()]

split_counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    labels=labels,
)
plt.title("Data Splits")
plt.tight_layout()
plt.savefig(PLOT_DIR / "data_split.svg")
plt.show()

In [ ]:
plt.figure(figsize=(18, 6))
order = ["neither", "fire_only", "smoke_only", "both"]
ax = (df.groupby("split")["category"].value_counts(normalize=True).unstack()
      .reindex(columns=order)
      .reindex(["train", "val", "test"])
      .plot(kind="bar", stacked=True)
      )

ax.legend(title="Category", labels=["Neither", "Fire Only", "Smoke Only", "Both"], bbox_to_anchor=(1.3, 0.2), loc="lower right")

split_sizes = df.groupby("split").size().reindex(["train", "val", "test"])

for c in ax.containers:
    labels = [
        f"{round(v.get_height() * split_sizes.iloc[i]):,}"
        if v.get_height() > 0 else ""
        for i, v in enumerate(c)
    ]

    ax.bar_label(c, labels=labels, label_type="center")

plt.title("Category Distribution by Split")

plt.xlabel("Split")
plt.ylabel("Proportion")
plt.yticks(np.arange(0, 1.1, 0.1))

plt.savefig(PLOT_DIR / "category_distribution_by_split.svg")
plt.show()

In [ ]:
sizes_df = df["boxes"].explode().dropna().apply(pd.Series)

sizes_df[["xc", "yc", "width", "height"]] = pd.DataFrame(
    sizes_df["coordinates"].tolist(),
    index=sizes_df.index
)

sizes_df["path"] = df.loc[sizes_df.index, "path"].to_numpy()
sizes_df["area"] = sizes_df["width"] * sizes_df["height"]
sizes_df["split"] = df.loc[sizes_df.index, "split"].to_numpy()

sizes_df.to_csv(ANALYSIS_DIR / "box_sizes.csv", index=False)
sizes_df

In [ ]:
label_counts = sizes_df["category"].value_counts()

fire_count = label_counts.get(1, 0)
smoke_count = label_counts.get(0, 0)

labels = [f"Fire ({fire_count})", f"Smoke ({smoke_count})"]

plt.figure(figsize=(6, 6))

plt.pie(
    [fire_count, smoke_count],
    autopct="%1.1f%%",
    labels=labels
)

plt.savefig(PLOT_DIR / "label_distribution.svg")
plt.show()